# Reproducibilidad — por qué la semilla no es un detalle cosmético

**Unidad 4.d** · Acompaña a `Clase_11_DiferenciaEnDiferencias` · Notas: cap. 9

Hacemos un *bootstrap* del estimador de diferencia en diferencias de Card y Krueger,
primero sin fijar semilla y luego fijándola.

Sin semilla, **dos corridas del mismo código sobre los mismos datos entregan intervalos
de confianza distintos**. El problema no es que la diferencia sea grande —no lo es—, sino
que hace imposible que alguien más verifique el número reportado, y que ni el propio
autor pueda reconstruirlo tres meses después.

In [1]:
import numpy as np
import pandas as pd

REPLICAS = 2000

datos = pd.read_csv("../../Clase_11_DiferenciaEnDiferencias/employment.csv")


def did(d):
    """Estimador de diferencia en diferencias por medias."""
    medias = d.groupby("state")[["total_emp_feb", "total_emp_nov"]].mean()
    cambio = medias["total_emp_nov"] - medias["total_emp_feb"]
    return cambio.loc[1] - cambio.loc[0]


puntual = did(datos)
print(f"Estimación puntual: {puntual:+.4f}   (réplicas del bootstrap: {REPLICAS})")

Estimación puntual: +2.7500   (réplicas del bootstrap: 2000)


In [2]:
def bootstrap(d, replicas=REPLICAS, generador=None):
    """Bootstrap por establecimiento. Si `generador` es None, usa el estado global."""
    n = len(d)
    if generador is None:
        sortear = lambda k: np.random.randint(0, k, k)
    else:
        sortear = lambda k: generador.integers(0, k, k)

    estimaciones = np.empty(replicas)
    for r in range(replicas):
        estimaciones[r] = did(d.iloc[sortear(n)])
    return estimaciones


def resumir(e):
    return {
        "ee": e.std(ddof=1),
        "ic_inf": np.percentile(e, 2.5),
        "ic_sup": np.percentile(e, 97.5),
    }


def imprimir(etiqueta, r):
    print(f"  {etiqueta:12s} ee = {r['ee']:.4f}   "
          f"IC 95 % = [{r['ic_inf']:+.4f}, {r['ic_sup']:+.4f}]")

## Sin fijar semilla — dos corridas del mismo código

In [3]:
a = resumir(bootstrap(datos))
b = resumir(bootstrap(datos))

imprimir("corrida 1", a)
imprimir("corrida 2", b)

print(f"\n  Diferencia en el ee             : {abs(a['ee'] - b['ee']):.4f}")
print(f"  Diferencia en el límite inferior: {abs(a['ic_inf'] - b['ic_inf']):.4f}")
print("\n  Los dos resultados son del mismo código y de los mismos datos.")
print("  Ninguno es 'el' resultado.")

  corrida 1    ee = 1.3107   IC 95 % = [+0.2703, +5.3449]
  corrida 2    ee = 1.3515   IC 95 % = [+0.0692, +5.4091]

  Diferencia en el ee             : 0.0408
  Diferencia en el límite inferior: 0.2012

  Los dos resultados son del mismo código y de los mismos datos.
  Ninguno es 'el' resultado.


## Con semilla fija — dos corridas del mismo código

In [4]:
c = resumir(bootstrap(datos, generador=np.random.default_rng(20260802)))
d = resumir(bootstrap(datos, generador=np.random.default_rng(20260802)))

imprimir("corrida 1", c)
imprimir("corrida 2", d)

print(f"\n  Idénticas: {np.isclose(c['ee'], d['ee']) and np.isclose(c['ic_inf'], d['ic_inf'])}")

  corrida 1    ee = 1.3566   IC 95 % = [+0.1358, +5.4277]
  corrida 2    ee = 1.3566   IC 95 % = [+0.1358, +5.4277]

  Idénticas: True


## Cómo se hace bien

### 1. Un generador explícito, pasado como argumento

```python
generador = np.random.default_rng(20260802)
```

En lugar de `np.random.seed()`, que modifica un **estado global** y hace que el resultado
dependa del **orden en que se ejecutaron las celdas** del cuaderno. Es la causa más común
de un cuaderno que no se puede reproducir *aunque tenga semilla*.

### 2. La semilla se declara en el texto del trabajo, no sólo en el código

### 3. Si el resultado cambia de manera relevante al cambiar la semilla, el problema no es la semilla

Son pocas réplicas o poca muestra. Compruébalo:

In [5]:
print(f"{'réplicas':>10s} {'ee':>10s} {'IC inferior':>14s}")
for r in [100, 500, 2000, 5000]:
    e = resumir(bootstrap(datos, replicas=r, generador=np.random.default_rng(1)))
    print(f"{r:10d} {e['ee']:10.4f} {e['ic_inf']:14.4f}")

print("\nAl aumentar las réplicas el resultado se estabiliza: la dispersión")
print("entre corridas era ruido de simulación, no incertidumbre de los datos.")

  réplicas         ee    IC inferior
       100     1.1552         0.6421
       500     1.2926         0.2619
      2000     1.3327         0.1476
      5000     1.3217         0.1664

Al aumentar las réplicas el resultado se estabiliza: la dispersión
entre corridas era ruido de simulación, no incertidumbre de los datos.


### 4. La versión de la biblioteca también importa

In [6]:
print(f"numpy {np.__version__}")
print()
print("El algoritmo generador puede cambiar entre versiones, y por eso la")
print("versión va en requirements.txt. Para un trabajo propio se fija con ==,")
print("no con >=: es la diferencia entre «corre» y «da el mismo número».")

numpy 1.26.4

El algoritmo generador puede cambiar entre versiones, y por eso la
versión va en requirements.txt. Para un trabajo propio se fija con ==,
no con >=: es la diferencia entre «corre» y «da el mismo número».


## Comprobación adicional

El error estándar del *bootstrap* debería parecerse al error estándar agrupado de la
regresión, que calculamos en
[`Replicar_Card_Krueger.ipynb`](../04_Verificar_resultados/Replicar_Card_Krueger.ipynb).
Es otra instancia del principio de las **dos implementaciones independientes**.

In [7]:
import statsmodels.formula.api as smf

largo = datos.reset_index(names="tienda").melt(
    id_vars=["tienda", "state"],
    value_vars=["total_emp_feb", "total_emp_nov"],
    var_name="periodo",
    value_name="empleo",
)
largo["post"] = (largo["periodo"] == "total_emp_nov").astype(int)
largo["tratado"] = largo["state"]

reg = smf.ols("empleo ~ tratado + post + tratado:post", data=largo).fit()
clu = reg.get_robustcov_results(cov_type="cluster", groups=largo["tienda"])

print(f"  ee por bootstrap        : {c['ee']:.4f}")
print(f"  ee agrupado por tienda  : {clu.bse[3]:.4f}")
print(f"  diferencia relativa     : {100 * abs(c['ee'] - clu.bse[3]) / clu.bse[3]:.1f} %")
print()
print("Dos métodos distintos que llegan casi al mismo número: es evidencia")
print("de que ninguno de los dos está mal implementado.")

  ee por bootstrap        : 1.3566
  ee agrupado por tienda  : 1.3386
  diferencia relativa     : 1.3 %

Dos métodos distintos que llegan casi al mismo número: es evidencia
de que ninguno de los dos está mal implementado.


## Y ahora, la lista

La actividad se cierra recorriendo
[`lista_de_verificacion.md`](lista_de_verificacion.md), que es la lista que hay que pasar
antes de **cada entrega** del semestre. Sus siete bloques corresponden, uno a uno, a
errores que las actividades de esta carpeta reproducen.

### Un punto que merece énfasis aparte

**Los modelos generativos producen referencias inexistentes con formato impecable.**
Autores que existen, revista que existe, título verosímil, año plausible — y el artículo
no existe. No hay forma de detectarlo leyendo la cita.

Toda referencia se abre y se verifica contra la fuente. Sin excepción.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las demás actividades.